# IMPORT

In [64]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import csv
from collections import Counter

# DEFINE FOLDERS

In [55]:
DATASET_INPUT_FOLDER = Path("correlated_data_csv")
PLOT_FOLDER = Path("plots")
MISSING_STATION_DATA_FOLDER = Path("missing_station_data")

folders = [DATASET_INPUT_FOLDER, PLOT_FOLDER, MISSING_STATION_DATA_FOLDER]
for folder in folders:
    folder.mkdir(parents=True, exist_ok=True)

# LOAD DATA

In [56]:
df = pd.read_csv(f"{DATASET_INPUT_FOLDER}/clean_data.csv")
df.head(5)

,station_code,latitude,longitude,capacity,total_bikes_now,total_bikes_taken,timestamp,temperature,wind_speed,wind_direction,weather_code,is_day,hour,dayofweek,is_weekend,sin_hour,cos_hour
0,14111,48.837526,2.336035,25,2,22,2025-11-04 04:00:00,10.1,9.5,217.0,3,0,4,1,0,0.866025,0.5
1,32017,48.936269,2.358867,22,6,16,2025-11-04 04:00:00,10.1,9.5,217.0,3,0,4,1,0,0.866025,0.5
2,11104,48.855908,2.392571,20,2,18,2025-11-04 04:00:00,10.1,9.5,217.0,3,0,4,1,0,0.866025,0.5
3,5110,48.843893,2.351966,23,7,15,2025-11-04 04:00:00,10.1,9.5,217.0,3,0,4,1,0,0.866025,0.5
4,6108,48.847082,2.321375,17,4,11,2025-11-04 04:00:00,10.1,9.5,217.0,3,0,4,1,0,0.866025,0.5


# CONSTANTS

In [57]:
EXPECTED_MINUTES = [0, 15, 30, 45]
EXPECTED_SLOTS_PER_DAY = 24 * 4  # 96

# MISSING DATA COMPUTATION PER STATION

In [62]:
def compute_missing_data(station_id, df):
    df_s = df[df["station_code"] == station_id].copy()

    # Ensure timestamp is datetime
    df_s["timestamp"] = pd.to_datetime(df_s["timestamp"])
    df_s["date"] = df_s["timestamp"].dt.date
    df_s["hour"] = df_s["timestamp"].dt.hour
    df_s["minute"] = df_s["timestamp"].dt.minute

    records = []

    for date, day_df in df_s.groupby("date"):
        # Count how many rows exist for each (hour, minute)
        counts = Counter(zip(day_df["hour"], day_df["minute"]))

        for hour in range(24):
            for minute in EXPECTED_MINUTES:
                records.append({
                    "station_code": station_id,
                    "date": date,
                    "hour": hour,
                    "minute": minute,
                    "present": counts.get((hour, minute), 0)  # number of rows at that time
                })

    return pd.DataFrame(records)

# COMPUTE MISSING DATA FOR ALL STATIONS

In [ ]:
all_missing = []

for station_id in df["station_code"].unique():
    all_missing.append(compute_missing_data(station_id, df))

missing_df = pd.concat(all_missing, ignore_index=True)
display(missing_df.head())
missing_df.describe()

,station_code,date,hour,minute,present
0,14111,2025-11-04,0,0,0
1,14111,2025-11-04,0,15,0
2,14111,2025-11-04,0,30,0
3,14111,2025-11-04,0,45,0
4,14111,2025-11-04,1,0,0


# SAVE IT

In [66]:
missing_df.to_csv(f"{MISSING_STATION_DATA_FOLDER}/missing_data_detailed.csv", index=False)
print(f"Saved: {MISSING_STATION_DATA_FOLDER}/missing_data_detailed.csv")

Saved: missing_station_data/missing_data_detailed.csv


# STATION SUMMARY

In [68]:
station_summary = (
    missing_df
    .groupby("station_code")["present"]
    .agg(
        total_slots="count",
        present_slots="sum"
    )
)

station_summary["missing_slots"] = (
    station_summary["total_slots"] - station_summary["present_slots"]
)

station_summary["missing_ratio"] = (
    station_summary["missing_slots"] / station_summary["total_slots"]
)

display(station_summary.sort_values("missing_ratio", ascending=False).head())
display(station_summary.describe())

,total_slots,present_slots,missing_slots,missing_ratio
station_code,,,,
35010,96,3,93,0.968750
43203,192,77,115,0.598958
8203,4800,2510,2290,0.477083
15056,4800,2538,2262,0.471250
12105,4800,2574,2226,0.463750


,total_slots,present_slots,missing_slots,missing_ratio
count,1497.000000,1497.000000,1497.000000,1497.000000
mean,4784.865731,3985.734803,799.130929,0.167841
std,215.675977,202.070644,89.553849,0.029314
min,96.000000,3.000000,93.000000,0.163750
25%,4800.000000,4003.000000,792.000000,0.165000
50%,4800.000000,4005.000000,795.000000,0.165625
75%,4800.000000,4008.000000,797.000000,0.166042
max,4800.000000,4014.000000,2290.000000,0.968750


# SAVE IT

In [69]:
station_summary.to_csv(f"{MISSING_STATION_DATA_FOLDER}/missing_data_by_station.csv")
print(f"Saved: {MISSING_STATION_DATA_FOLDER}/missing_data_by_station.csv")

Saved: missing_station_data/missing_data_by_station.csv


# MISSING DATA BY TIMESTAMP

In [70]:
time_summary = (
    missing_df
    .groupby(["hour", "minute"])["present"]
    .mean()
    .reset_index()
)

time_summary["missing_ratio"] = 1 - time_summary["present"]
time_summary.sort_values("missing_ratio", ascending=False).head()

,hour,minute,present,missing_ratio
29,7,15,0.690916,0.309084
41,10,15,0.695821,0.304179
57,14,15,0.714062,0.285938
70,17,30,0.714732,0.285268
63,15,45,0.723926,0.276074


# SAVE IT

In [71]:
time_summary.to_csv(f"{MISSING_STATION_DATA_FOLDER}/missing_data_by_timestamp.csv", index=False)
print(f"Saved: {MISSING_STATION_DATA_FOLDER}/missing_data_by_timestamp.csv")

Saved: missing_station_data/missing_data_by_timestamp.csv


# MISSING COMPLETE DAYS

In [72]:
daily_summary = (
    missing_df
    .groupby(["station_code", "date"])["present"]
    .sum()
    .reset_index()
)

daily_summary["complete_day_missing"] = (daily_summary["present"] == 0)
daily_summary[daily_summary["complete_day_missing"]].head()

,station_code,date,present,complete_day_missing


# SAVE IT

In [73]:
daily_summary.to_csv(f"{MISSING_STATION_DATA_FOLDER}/missing_data_by_day.csv", index=False)
print(f"Saved: {MISSING_STATION_DATA_FOLDER}/missing_data_by_day.csv")

Saved: missing_station_data/missing_data_by_day.csv


# DUPLICATES IN THE DATA

In [74]:
# Check duplicates: same station, same timestamp
duplicates = df[df.duplicated(subset=["station_code", "timestamp"], keep=False)]
print(f"Number of duplicate rows: {len(duplicates)}")

# If you want to see which stations have duplicates
dup_counts = duplicates.groupby("station_code").size().sort_values(ascending=False)
print(dup_counts)


Number of duplicate rows: 19678
station_code
17011    14
21011    14
21021    14
21019    14
21017    14
         ..
16006    12
22209    11
32308     2
40018     2
22508     2
Length: 1498, dtype: int64


# PUT THE MISSING DATA IN PLOTS

In [75]:
def plot_missing_data(station_id, df, save_path=None):
    df_s = df[df["station_code"] == station_id].copy()

    # Snap timestamps to nearest 15min to reduce misalignment issues
    df_s["timestamp"] = pd.to_datetime(df_s["timestamp"]).dt.round("15min")

    # Extract date and time
    df_s["date"] = df_s["timestamp"].dt.date
    df_s["time"] = df_s["timestamp"].dt.time

    # Mark presence
    df_s["present"] = 1

    # Aggregate duplicates
    df_s_agg = df_s.groupby(["date", "time"])["present"].max().reset_index()

    # Pivot
    heatmap_df = df_s_agg.pivot(index="date", columns="time", values="present").fillna(0)

    # Convert time columns to numeric hours for X-axis
    x_labels = [pd.Timestamp.combine(pd.Timestamp.today(), t).hour + pd.Timestamp.combine(pd.Timestamp.today(), t).minute/60 for t in heatmap_df.columns]

    # Plot
    plt.figure(figsize=(18, max(6, len(heatmap_df)/5)))
    plt.imshow(heatmap_df, aspect="auto", cmap="Greens", interpolation="none")
    plt.xticks(ticks=np.linspace(0, len(x_labels)-1, 12), 
               labels=[f"{int(x)}:00" for x in np.linspace(0, 24, 12)], rotation=45)
    plt.yticks(range(len(heatmap_df.index)), [str(d) for d in heatmap_df.index])
    plt.xlabel("Time of day")
    plt.ylabel("Date")
    plt.title(f"Station {station_id} — Missing data pattern")
    plt.colorbar(label="Presence (1=present, 0=missing)")
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path)
        plt.close()
    else:
        plt.show()

# SAVE IT

In [77]:
for station_id in df["station_code"].unique():
    plot_missing_data(station_id, df, f"{PLOT_FOLDER}/station_{station_id}.png")

print("All station plots generated.")

C:\Users\emirh\AppData\Local\Temp\ipykernel_171148\1717140192.py:25: UserWarning: Attempting to set identical low and high xlims makes transformation singular; automatically expanding.
  plt.imshow(heatmap_df, aspect="auto", cmap="Greens", interpolation="none")
C:\Users\emirh\AppData\Local\Temp\ipykernel_171148\1717140192.py:25: UserWarning: Attempting to set identical low and high ylims makes transformation singular; automatically expanding.
  plt.imshow(heatmap_df, aspect="auto", cmap="Greens", interpolation="none")
C:\Users\emirh\AppData\Local\Temp\ipykernel_171148\1717140192.py:25: UserWarning: Attempting to set identical low and high xlims makes transformation singular; automatically expanding.
  plt.imshow(heatmap_df, aspect="auto", cmap="Greens", interpolation="none")
C:\Users\emirh\AppData\Local\Temp\ipykernel_171148\1717140192.py:25: UserWarning: Attempting to set identical low and high ylims makes transformation singular; automatically expanding.
  plt.imshow(heatmap_df, aspe

All station plots generated.
